# EcoCrop Poland Dataset

## 1. Configuration

In [3]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

SOURCE_PATH = Path("data/EcoCrop_DB.csv")
OUTPUT_PATH = Path("data/EcoCrop_Poland.csv")
SOURCE_ENCODING = "latin1"

POLAND_CLIMATE_PATTERN = (
    r"temperate oceanic \(Do\)|"
    r"temperate continental \(Dc\)|"
    r"temperate with humid winters \(Df\)"
)
GROWING_SEASON_OPTIMUM_C = (10, 20)
GROWING_SEASON_LIMIT_C = (10, 25)

## 2. Source data

In [2]:
source_df = pd.read_csv(
    SOURCE_PATH,
    encoding=SOURCE_ENCODING,
    na_values=["NA", ""],
)
working_df = source_df.copy(deep=True)

assert len(source_df) > 0
assert source_df.columns.is_unique

print(f"Source shape: {source_df.shape}")
display(source_df.head(3))

Source shape: (2568, 55)


,EcoPortCode,ScientificName,AUTH,FAMNAME,SYNO,COMNAME,LIFO,HABI,LISPA,PHYS,CAT,PLAT,TOPMN,TOPMX,TMIN,TMAX,ROPMN,ROPMX,RMIN,RMAX,PHOPMN,PHOPMX,PHMIN,PHMAX,LATOPMN,LATOPMX,LATMN,LATMX,ALTMX,LIOPMN,LIOPMX,LIMN,LIMX,DEP,DEPR,TEXT,TEXTR,FER,FERR,TOX,TOXR,SAL,SALR,DRA,DRAR,KTMPR,KTMP,PHOTO,CLIZ,ABITOL,ABISUS,INTRI,PROSY,GMIN,GMAX
0,289,Abelmoschus esculentus,(L.) Moench,Magnoliopsida:Dilleniidae:Malvales:Malvaceae,Hibiscus esculentus L.,"abelmoskus, america-neri, bakhua mun, bamia, bamija, bamya, bandakai, bende'...",herb,erect,annual,single stem,vegetables,grown on large scale,20.0,30.0,12.0,35.0,600.0,1200.0,300.0,2500.0,5.5,7.0,4.5,8.7,NaN,NaN,35.0,40.0,1000.0,clear skies,clear skies,very bright,cloudy skies,shallow (20-50 cm),shallow (20-50 cm),"heavy, medium, light, organic","heavy, medium, light",high,moderate,NaN,NaN,low (<4 dS/m),low (<4 dS/m),well (dry spells),well (dry spells),NaN,NaN,short day (<12 hours),"tropical wet & dry (Aw), tropical wet (Ar), steppe or semiarid (Bs), subtrop...",NaN,NaN,NaN,NaN,50.0,180.0
1,290,Abelmoschus manihot,(L.) Medic.,Magnoliopsida:Dilleniidae:Malvales:Malvaceae,"Hibiscus manihot L. (1753), Abelmoschus manihot ssp. manihot [(L)]; Medikus","neka (Simbo), bele (Fiji), pele (Tonga, Tuvalu), aibika, island cabbage, bae...",shrub,erect,"annual, perennial",NaN,"vegetables, ornamentals/turf, medicinals & aromatic",grown on small scale,22.0,30.0,16.0,36.0,1200.0,2000.0,1000.0,3000.0,5.5,7.0,5.0,8.0,NaN,NaN,NaN,NaN,1200.0,very bright,clear skies,very bright,cloudy skies,medium (50-150 cm),medium (50-150 cm),"medium, organic","heavy, medium, light",high,moderate,NaN,NaN,low (<4 dS/m),low (<4 dS/m),well (dry spells),well (dry spells),NaN,NaN,long day (>14 hours),"tropical wet & dry (Aw), tropical wet (Ar)",NaN,NaN,NaN,NaN,365.0,365.0
2,291,Abelmoschus moschatus,Medic.,Magnoliopsida:Dilleniidae:Malvales:Malvaceae,Hibiscus abelmoschus L. (1753).,"abelmosk, musk mallow, mushkdan, muskdana, kasturi bhendi, gukhia korai, kas...","herb, sub-shrub",prostrate/procumbent/semi-erect,"annual, biennial, perennial","deciduous, multi stem","ornamentals/turf, medicinals & aromatic",NaN,20.0,30.0,10.0,35.0,1000.0,1400.0,900.0,1500.0,5.5,7.0,5.0,8.0,NaN,NaN,23.0,29.0,900.0,cloudy skies,very bright,light shade,very bright,medium (50-150 cm),shallow (20-50 cm),"medium, organic","medium, organic",high,moderate,NaN,NaN,low (<4 dS/m),low (<4 dS/m),well (dry spells),well (dry spells),NaN,NaN,NaN,tropical wet & dry (Aw),NaN,NaN,can become a weed,NaN,0.0,0.0


## 3. Climate filter

In [3]:
climate_mask = working_df["CLIZ"].fillna("").str.contains(
    POLAND_CLIMATE_PATTERN,
    regex=True,
)

optimal_temperature_mask = (
    working_df["TOPMN"].le(GROWING_SEASON_OPTIMUM_C[1])
    & working_df["TOPMX"].ge(GROWING_SEASON_OPTIMUM_C[0])
)

limit_temperature_mask = (
    working_df["TMIN"].le(GROWING_SEASON_LIMIT_C[0])
    & working_df["TMAX"].ge(GROWING_SEASON_LIMIT_C[1])
)

poland_mask = climate_mask & optimal_temperature_mask & limit_temperature_mask
poland_df = working_df.loc[poland_mask].copy()

print(f"Climate match: {climate_mask.sum():,}")
print(f"Poland dataset: {len(poland_df):,}")

Climate match: 767
Poland dataset: 480


## 4. Dataset preparation

In [4]:
poland_df["FAMILY"] = (
    poland_df["FAMNAME"]
    .fillna("Unknown")
    .astype(str)
    .str.split(":")
    .str[-1]
    .str.strip()
)
poland_df["PLANT_ID"] = pd.to_numeric(
    poland_df["EcoPortCode"].astype(str).str.extract(r"(\d+)$")[0],
    errors="coerce",
).astype("Int64")

column_map = {
    "PLANT_ID": "plant_id",
    "ScientificName": "scientific_name",
    "COMNAME": "common_names",
    "FAMILY": "family",
    "LIFO": "life_form",
    "HABI": "growth_habit",
    "LISPA": "life_span",
    "PHYS": "physiology",
    "CAT": "category",
    "PLAT": "cultivation_scale",
    "TOPMN": "optimal_temperature_min_c",
    "TOPMX": "optimal_temperature_max_c",
    "TMIN": "absolute_temperature_min_c",
    "TMAX": "absolute_temperature_max_c",
    "ROPMN": "optimal_rainfall_min_mm",
    "ROPMX": "optimal_rainfall_max_mm",
    "RMIN": "absolute_rainfall_min_mm",
    "RMAX": "absolute_rainfall_max_mm",
    "PHOPMN": "optimal_ph_min",
    "PHOPMX": "optimal_ph_max",
    "PHMIN": "absolute_ph_min",
    "PHMAX": "absolute_ph_max",
    "LIOPMN": "optimal_light_min",
    "LIOPMX": "optimal_light_max",
    "LIMN": "absolute_light_min",
    "LIMX": "absolute_light_max",
    "DEP": "optimal_soil_depth",
    "DEPR": "absolute_soil_depth",
    "TEXT": "optimal_soil_texture",
    "TEXTR": "absolute_soil_texture",
    "FER": "optimal_soil_fertility",
    "FERR": "absolute_soil_fertility",
    "SAL": "optimal_salinity",
    "SALR": "absolute_salinity",
    "DRA": "optimal_drainage",
    "DRAR": "absolute_drainage",
    "KTMPR": "killing_temperature_min_c",
    "KTMP": "killing_temperature_max_c",
    "PHOTO": "photoperiod",
    "CLIZ": "climate_zones",
    "ABITOL": "abiotic_tolerances",
    "ABISUS": "abiotic_susceptibilities",
    "INTRI": "invasiveness_risk",
    "PROSY": "production_system",
    "GMIN": "growing_period_min_days",
    "GMAX": "growing_period_max_days",
}

poland_dataset = (
    poland_df[list(column_map)]
    .rename(columns=column_map)
    .drop_duplicates(subset=["scientific_name"], keep="first")
    .sort_values(["family", "scientific_name"], na_position="last")
    .reset_index(drop=True)
)

## 5. Validation

In [5]:
required_columns = [
    "plant_id",
    "scientific_name",
    "family",
    "optimal_temperature_min_c",
    "optimal_temperature_max_c",
    "absolute_temperature_min_c",
    "absolute_temperature_max_c",
    "climate_zones",
]

assert poland_dataset["scientific_name"].is_unique
assert poland_dataset[required_columns].notna().all().all()
assert poland_dataset["climate_zones"].str.contains(
    POLAND_CLIMATE_PATTERN,
    regex=True,
).all()
assert len(source_df) == len(working_df)

quality = pd.DataFrame({
    "dtype": poland_dataset.dtypes.astype(str),
    "non_null": poland_dataset.notna().sum(),
    "missing_pct": (poland_dataset.isna().mean() * 100).round(1),
    "unique": poland_dataset.nunique(),
})

display(quality)
display(poland_dataset.head(20))

,dtype,non_null,missing_pct,unique
plant_id,Int64,480,0.0,480
scientific_name,str,480,0.0,480
common_names,str,354,26.2,354
family,object,480,0.0,80
life_form,str,480,0.0,14
growth_habit,str,463,3.5,9
life_span,str,479,0.2,7
physiology,str,385,19.8,18
category,str,477,0.6,131
cultivation_scale,str,279,41.9,8


,plant_id,scientific_name,common_names,family,life_form,growth_habit,life_span,physiology,category,cultivation_scale,optimal_temperature_min_c,optimal_temperature_max_c,absolute_temperature_min_c,absolute_temperature_max_c,optimal_rainfall_min_mm,optimal_rainfall_max_mm,absolute_rainfall_min_mm,absolute_rainfall_max_mm,optimal_ph_min,optimal_ph_max,absolute_ph_min,absolute_ph_max,optimal_light_min,optimal_light_max,absolute_light_min,absolute_light_max,optimal_soil_depth,absolute_soil_depth,optimal_soil_texture,absolute_soil_texture,optimal_soil_fertility,absolute_soil_fertility,optimal_salinity,absolute_salinity,optimal_drainage,absolute_drainage,killing_temperature_min_c,killing_temperature_max_c,photoperiod,climate_zones,abiotic_tolerances,abiotic_susceptibilities,invasiveness_risk,production_system,growing_period_min_days,growing_period_max_days
0,2737,Acanthus mollis,acanthus,Acanthaceae,herb,erect,perennial,NaN,"ornamentals/turf, medicinals & aromatic",grown on small scale,15.0,30.0,10.0,38.0,500.0,800.0,400.0,1000.0,5.0,7.0,5.5,6.5,light shade,cloudy skies,light shade,very bright,medium (50-150 cm),shallow (20-50 cm),medium,"medium, light",high,moderate,low (<4 dS/m),low (<4 dS/m),well (dry spells),"well (dry spells), excessive (dry/moderately dry)",NaN,NaN,NaN,"subtropical dry summer (Cs), temperate oceanic (Do), temperate continental (Dc)",NaN,NaN,NaN,home garden,0.0,0.0
1,2743,Acer caesium,"Indian maple, bordo da India",Aceraceae,tree,erect,perennial,"deciduous, single stem",forest/wood,NaN,15.0,21.0,7.0,30.0,1000.0,1800.0,660.0,2150.0,4.5,5.5,4.0,6.5,very bright,clear skies,very bright,light shade,deep (>>150 cm),medium (50-150 cm),medium,"heavy, medium, light",high,moderate,low (<4 dS/m),low (<4 dS/m),well (dry spells),"well (dry spells), excessive (dry/moderately dry)",-5.0,0.0,short day (<12 hours),"tropical wet (Ar), subtropical humid (Cf), subtropical dry summer (Cs), subt...",NaN,NaN,NaN,NaN,210.0,270.0
2,2756,Acer saccharum,"sugar maple, sugarwood, rock maple, hard maple, bird's-eye maple, erable a s...",Aceraceae,tree,erect,perennial,"deciduous, single stem",forest/wood,NaN,14.0,24.0,4.0,28.0,700.0,1300.0,500.0,1500.0,6.0,6.5,5.3,7.3,very bright,clear skies,very bright,cloudy skies,deep (>>150 cm),medium (50-150 cm),medium,medium,moderate,moderate,low (<4 dS/m),low (<4 dS/m),well (dry spells),well (dry spells),-18.0,-1.0,long day (>14 hours),"steppe or semiarid (Bs), temperate oceanic (Do), temperate continental (Dc),...",NaN,NaN,NaN,NaN,130.0,240.0
3,2779,Acorus calamus,"calamus, sweetflag, acore vrai, kalmus, bojo, kull, bach, makkalmoes, ikalam...",Acoraceae,herb,erect,perennial,NaN,medicinals & aromatic,grown on small scale,18.0,26.0,4.0,30.0,7000.0,2000.0,430.0,4200.0,5.5,6.5,5.0,7.5,very bright,clear skies,very bright,cloudy skies,medium (50-150 cm),shallow (20-50 cm),"heavy, medium, light","heavy, medium, light",low,low,low (<4 dS/m),low (<4 dS/m),poorly (saturated >50% of year),poorly (saturated >50% of year),NaN,NaN,"short day (<12 hours), neutral day (12-14 hours), long day (>14 hours)","tropical wet & dry (Aw), tropical wet (Ar), steppe or semiarid (Bs), subtrop...",NaN,NaN,NaN,NaN,90.0,365.0
4,2803,Actinidia arguta,"hardy kiwi, bower vine, dessert kiwi, cocktail kiwi, tara vine, yang-tao",Actinidiaceae,vine,climber/scrambler/scadent,perennial,"deciduous, single stem","fruits & nuts, environmental",NaN,12.0,25.0,5.0,30.0,900.0,1500.0,700.0,2000.0,5.0,6.5,4.5,7.0,cloudy skies,very bright,light shade,very bright,medium (50-150 cm),medium (50-150 cm),"medium, light, organic","medium, light, organic",high,moderate,low (<4 dS/m),low (<4 dS/m),well (dry spells),well (dry spells),-30.0,0.0,NaN,"subtropical humid (Cf), subtropical dry summer (Cs), subtropical dry winter ...",NaN,NaN,NaN,NaN,150.0,180.0
5,10370,Tetragonia tetragonioides,NaN,Aizoaceae,herb,prostrate/procumbent/semi-erect,"annual, biennial",multi stem,vegetables,grown on small scale,19.0,28.0,10.0,36.0,900.0,1200.0,700.0,3000.0,6.5,7.0,6.0,7

## 6. Export

In [6]:
poland_dataset.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(poland_dataset):,}")
print(f"Columns: {poland_dataset.shape[1]}")

Saved: data\EcoCrop_Poland.csv
Rows: 480
Columns: 46
